# 4. Диагностика: классы спроса и метрики

**Сквозной вопрос блока:** почему на нулях ломаются и модели, и метрики — и что с этим делать.

В этом ноутбуке:
- Что такое прерывистый спрос, примеры из M5;
- Классификация ADI / CV² (Syntetos–Boylan–Croston);
- Почему MAE «любит» нулевой прогноз;
- Метрики для прерывистого спроса (RMSSE, bias, PIS);
- Бейзлайны: ноль, Naive, SeasonalNaive, SES.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 4)

## 0. Данные: M5 CA_1

M5 Competition: Walmart, дневные продажи товаров. Подвыборка — магазин CA_1 (Калифорния), все 3049 товаров.
Формат: wide (d_1, d_2, ..., d_1913) → преобразуем в long.

In [ ]:
# Загрузка
sales = pd.read_csv('../data/sales_train_validation.csv')
calendar = pd.read_csv('../data/calendar.csv')

# Фильтр CA_1
sales_ca1 = sales[sales['store_id'] == 'CA_1'].copy()

print(f"Товаров в CA_1: {len(sales_ca1)}")
print(f"Дней: d_1 — d_{len([c for c in sales.columns if c.startswith('d_')])}")

In [ ]:
# Преобразование в long-формат
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in sales_ca1.columns if c.startswith('d_')]

df_long = sales_ca1.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='sales'
)

# Добавляем дату из calendar
df_long = df_long.merge(calendar[['d', 'date']], on='d', how='left')
df_long['date'] = pd.to_datetime(df_long['date'])
df_long = df_long.sort_values(['id', 'date']).reset_index(drop=True)

print(f"Long-формат: {len(df_long)} строк")
print(f"Период: {df_long['date'].min()} — {df_long['date'].max()}")

## 1. Что такое прерывистый спрос

Прерывистый (intermittent) спрос: много нулей между ненулевыми значениями. Типично для:
- редких товаров (запчасти, специализированная продукция);
- новых товаров;
- товаров с высокой ценой или длинным циклом покупки.

**Проблема:** классические модели (ETS, ARIMA) предполагают непрерывный ряд, на нулях дают плохие результаты.

In [ ]:
# Примеры рядов по одному из каждого класса (выберем позже)
# Пока визуализируем первые 4 товара
sample_ids = df_long['id'].unique()[:4]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

for ax, item_id in zip(axes, sample_ids):
    item_data = df_long[df_long['id'] == item_id]
    ax.plot(item_data['date'], item_data['sales'], linewidth=0.8)
    ax.set_ylabel('Продажи')
    ax.set_title(f'{item_id}')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Доля нулей по товарам
zero_fraction = df_long.groupby('id')['sales'].apply(lambda x: (x == 0).mean())

plt.figure(figsize=(10, 4))
plt.hist(zero_fraction, bins=50, edgecolor='black')
plt.xlabel('Доля нулей')
plt.ylabel('Число товаров')
plt.title('Распределение доли нулей по товарам CA_1')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Медианная доля нулей: {zero_fraction.median():.2f}")
print(f"Товаров с >50% нулей: {(zero_fraction > 0.5).sum()} из {len(zero_fraction)}")

## 2. Классификация ADI / CV²

Syntetos, Boylan, Croston (2005): два параметра характеризуют спрос:
- **ADI** (Average Demand Interval) = средний интервал между ненулевыми продажами;
- **CV²** = (σ / μ)² = квадрат коэффициента вариации ненулевых продаж.

Пороги (Syntetos & Boylan):
- ADI < 1.32 и CV² < 0.49 → **smooth** (гладкий);
- ADI ≥ 1.32 и CV² < 0.49 → **intermittent** (прерывистый);
- ADI < 1.32 и CV² ≥ 0.49 → **erratic** (неустойчивый);
- ADI ≥ 1.32 и CV² ≥ 0.49 → **lumpy** (комковатый).

In [ ]:
def classify_demand(series):
    """
    Классификация временного ряда по ADI и CV².

    Возвращает: {'adi': float, 'cv2': float, 'class': str}
    """
    # Интервалы между ненулевыми значениями
    nonzero_idx = np.where(series > 0)[0]

    if len(nonzero_idx) == 0:
        return {'adi': np.inf, 'cv2': np.inf, 'class': 'lumpy', 'zero_count': len(series)}

    if len(nonzero_idx) == 1:
        adi = len(series)
        cv2 = 0
    else:
        intervals = np.diff(nonzero_idx)
        adi = intervals.mean() if len(intervals) > 0 else len(series)

        nonzero_vals = series[nonzero_idx]
        if nonzero_vals.mean() > 0:
            cv2 = (nonzero_vals.std() / nonzero_vals.mean()) ** 2
        else:
            cv2 = 0

    # Классификация
    if adi < 1.32 and cv2 < 0.49:
        demand_class = 'smooth'
    elif adi >= 1.32 and cv2 < 0.49:
        demand_class = 'intermittent'
    elif adi < 1.32 and cv2 >= 0.49:
        demand_class = 'erratic'
    else:
        demand_class = 'lumpy'

    return {
        'adi': adi,
        'cv2': cv2,
        'class': demand_class,
        'zero_count': (series == 0).sum()
    }

In [ ]:
# Классификация всех товаров
classifications = []

for item_id in df_long['id'].unique():
    item_sales = df_long[df_long['id'] == item_id]['sales'].values
    result = classify_demand(item_sales)
    result['id'] = item_id
    classifications.append(result)

class_df = pd.DataFrame(classifications)

In [ ]:
# Распределение классов
class_counts = class_df['class'].value_counts()
print("Распределение классов спроса:\n")
print(class_counts)
print(f"\nДоли:\n{(class_counts / len(class_df) * 100).round(1)}")

In [ ]:
# Scatter ADI vs CV²
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'smooth': 'green', 'intermittent': 'blue', 'erratic': 'orange', 'lumpy': 'red'}

for cls in ['smooth', 'intermittent', 'erratic', 'lumpy']:
    subset = class_df[class_df['class'] == cls]
    ax.scatter(subset['adi'], subset['cv2'], alpha=0.5, label=cls, s=10, color=colors[cls])

ax.axvline(1.32, color='black', linestyle='--', linewidth=0.8)
ax.axhline(0.49, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('ADI (средний интервал между ненулевыми продажами)')
ax.set_ylabel('CV² (квадрат коэффициента вариации)')
ax.set_title('Классификация товаров по ADI / CV²')
ax.legend()
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Эффект агрегации

Агрегация по времени (день → неделя) уменьшает ADI и CV² → ряд может поменять класс.

In [ ]:
# Пример: один товар по дням и по неделям
example_id = class_df[class_df['class'] == 'intermittent']['id'].iloc[0]
example_data = df_long[df_long['id'] == example_id].copy()

# Недельная агрегация
example_data['week'] = example_data['date'].dt.to_period('W')
weekly = example_data.groupby('week')['sales'].sum().values

# Классификация
daily_class = classify_demand(example_data['sales'].values)
weekly_class = classify_demand(weekly)

print(f"Товар: {example_id}")
print(f"Дневной уровень: ADI={daily_class['adi']:.2f}, CV²={daily_class['cv2']:.2f}, класс={daily_class['class']}")
print(f"Недельный уровень: ADI={weekly_class['adi']:.2f}, CV²={weekly_class['cv2']:.2f}, класс={weekly_class['class']}")

## 3. Метрики

### Почему MAE «любит» нулевой прогноз

MAE минимизируется медианой. Если >50% продаж = 0, медиана = 0 → MAE минимален при нулевом прогнозе.
Такой прогноз бесполезен для управления запасами.

In [ ]:
# Демонстрация на синтетике
np.random.seed(42)
intermittent_series = np.array([0]*70 + [1]*10 + [2]*10 + [3]*10)
np.random.shuffle(intermittent_series)

# Прогнозы
forecasts = {
    'Нулевой': np.zeros(len(intermittent_series)),
    'Среднее': np.full(len(intermittent_series), intermittent_series.mean()),
    'Медиана': np.full(len(intermittent_series), np.median(intermittent_series))
}

# Метрики
for name, fc in forecasts.items():
    mae = np.abs(intermittent_series - fc).mean()
    rmse = np.sqrt(((intermittent_series - fc) ** 2).mean())
    print(f"{name:12s}: MAE={mae:.3f}, RMSE={rmse:.3f}")

print(f"\nМедиана ряда: {np.median(intermittent_series)}")
print("→ MAE минимален при нулевом прогнозе, но он бесполезен.")

### RMSSE, bias, PIS

- **RMSSE** (Root Mean Squared Scaled Error): RMSE, масштабированная наивным прогнозом. RMSSE < 1 → лучше naive.
- **Bias** = среднее (факт − прогноз). Положительный bias = недопрогноз, отрицательный = перепрогноз.
- **CFE** (Cumulative Forecast Error) = сумма ошибок во времени.
- **PIS** (Periods In Stock) = доля периодов, когда прогноз ≥ факт (товар в наличии).

In [ ]:
def rmsse(y_true, y_pred, y_train):
    """RMSSE: RMSE масштабированная naive baseline."""
    mse = ((y_true - y_pred) ** 2).mean()
    naive_errors = np.diff(y_train)
    scale = (naive_errors ** 2).mean()

    if scale == 0:
        return np.nan

    return np.sqrt(mse / scale)

def bias_metric(y_true, y_pred):
    """Bias: среднее (факт - прогноз)."""
    return (y_true - y_pred).mean()

def pis(y_true, y_pred):
    """PIS: доля периодов с прогнозом >= факт."""
    return (y_pred >= y_true).mean()

print("Функции метрик определены.")

## 4. Бейзлайны

Оценим простые методы на всех товарах CA_1. Горизонт H=28 (как в M5), train/test split в −28 дней.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, HistoricAverage, SimpleExponentialSmoothing

H = 28

# Подготовка: последние H дней — тест
df_long = df_long.sort_values(['id', 'date'])

train = df_long.groupby('id').apply(lambda x: x.iloc[:-H]).reset_index(drop=True)
test = df_long.groupby('id').apply(lambda x: x.iloc[-H:]).reset_index(drop=True)

print(f"Train: {len(train)}, Test: {len(test)}")

In [ ]:
# Для statsforecast нужен формат: unique_id, ds, y
train_sf = train[['id', 'date', 'sales']].rename(columns={'id': 'unique_id', 'date': 'ds', 'sales': 'y'})
test_sf = test[['id', 'date', 'sales']].rename(columns={'id': 'unique_id', 'date': 'ds', 'sales': 'y'})

In [ ]:
# Модели
models = [
    Naive(),
    SeasonalNaive(season_length=7),
    HistoricAverage(),
    SimpleExponentialSmoothing(alpha=0.2)
]

sf = StatsForecast(models=models, freq='D', n_jobs=1)

In [ ]:
# Прогноз
sf.fit(train_sf)
forecast = sf.predict(h=H)

In [ ]:
# Добавляем класс спроса
forecast = forecast.reset_index().merge(class_df[['id', 'class']], left_on='unique_id', right_on='id', how='left')
test_sf = test_sf.merge(class_df[['id', 'class']], left_on='unique_id', right_on='id', how='left')

In [ ]:
# Метрики по классам
results = []

for model_name in ['Naive', 'SeasonalNaive', 'HistoricAverage', 'SimpleExponentialSmoothing']:
    for cls in ['smooth', 'intermittent', 'erratic', 'lumpy', 'all']:
        if cls == 'all':
            test_cls = test_sf
            fc_cls = forecast
        else:
            test_cls = test_sf[test_sf['class'] == cls]
            fc_cls = forecast[forecast['class'] == cls]

        if len(test_cls) == 0:
            continue

        y_true = test_cls.groupby('unique_id')['y'].apply(list).values
        y_pred = fc_cls.groupby('unique_id')[model_name].apply(list).values

        # Усреднённые метрики
        rmses = []
        biases = []
        pis_vals = []

        for yt, yp in zip(y_true, y_pred):
            yt = np.array(yt)
            yp = np.array(yp)

            if len(yt) != len(yp):
                continue

            rmses.append(np.sqrt(((yt - yp) ** 2).mean()))
            biases.append((yt - yp).mean())
            pis_vals.append((yp >= yt).mean())

        results.append({
            'Модель': model_name,
            'Класс': cls,
            'RMSE': np.mean(rmses),
            'Bias': np.mean(biases),
            'PIS': np.mean(pis_vals),
            'Товаров': len(y_true)
        })

results_df = pd.DataFrame(results)

In [ ]:
# Визуализация RMSE по классам
pivot = results_df[results_df['Класс'] != 'all'].pivot(index='Модель', columns='Класс', values='RMSE')

pivot.plot(kind='bar', figsize=(12, 5))
plt.ylabel('RMSE')
plt.title('Бейзлайны: RMSE по классам спроса')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Класс спроса')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
print("\nБейзлайны: метрики по классам спроса\n")
print(results_df[results_df['Класс'] != 'all'].to_string(index=False))

## Итог

- M5 CA_1: 3049 товаров, ~50% — intermittent/lumpy.
- Классификация ADI/CV² разделяет товары на 4 класса; агрегация меняет класс.
- MAE минимален при нулевом прогнозе для рядов с >50% нулей — метрика не подходит.
- RMSSE, bias, PIS — подходящие метрики для прерывистого спроса.
- Бейзлайны: SeasonalNaive(7) часто лучше Naive; SES хуже на прерывистых данных.

**Следующий ноутбук:** специализированные методы — Croston, SBA, TSB, ADIDA, IMAPA.